# Notebook da atividade prática - Programação assistida por inteligência artificial

- **Aluno:** Heitor Andrade
- **Curso:** Ciência da Computação - UDF
- **Disciplina:** Tendências em Ciência da Computação
- **Professora:** Kadidja Valéria
- **Unidade:** 2
- **Data:** 10/09/2026
- **Miniprojeto escolhido:** Jogo da Forca - nível intermediário
- **Ferramentas utilizadas nesta entrega:** Python, terminal e Codex como assistente de IA.


## Como usar este notebook

Este arquivo reúne o relatório e os **códigos completos e comentados**, com exemplos executáveis para comparar a base com a versão final. Ele funciona de forma independente: não precisa importar os arquivos `.py` do repositório.

1. Abra o arquivo no Jupyter, no suporte a notebooks da sua IDE ou envie o `.ipynb` ao Google Colab.
2. Selecione um ambiente Python 3.10 ou superior e execute as células de cima para baixo.
3. As demonstrações usam entradas programadas, e os testes rodam sem pedir digitação. Assim, **Executar tudo** consegue chegar ao final.
4. Para jogar pelo teclado, siga as instruções da seção **Partida interativa opcional**.

As saídas abaixo foram salvas após uma execução completa em um kernel novo. O código da atividade usa apenas a biblioteca padrão do Python; não há instalação de pacotes dentro do notebook.

**Adaptação para notebook:** a função da base foi renomeada para `jogar_base`, para não sobrescrever `jogar` da versão final. Os blocos `if __name__ == "__main__"` foram substituídos por células de execução. Os testes usam as funções definidas aqui, sem importar `forca.py`.


## 1. Objetivo e escolha do projeto

A proposta da atividade é usar a IA como apoio durante a programação, construir uma base e depois avaliar e melhorar o código. O trabalho segue o ciclo apresentado no material: contextualizar, avaliar criticamente, refatorar e decidir o que integrar.

Escolhi o jogo da forca porque ele permite trabalhar com repetição, condições, coleções e funções em um problema fácil de visualizar. Apesar de parecer simples, o jogo exige atenção a situações como repetir uma letra, digitar uma entrada inválida ou descobrir uma letra que aparece várias vezes na palavra.

O objetivo foi entregar um jogo de terminal que funcionasse e cujo código fosse fácil de explicar. A atividade foi desenvolvida com assistência de IA, incluindo implementação, revisão e execução dos testes. Não houve uma sessão separada com GitHub Copilot, Replit ou Colab; o ambiente utilizado foi o projeto local com Python.


## 2. Regras definidas

1. O programa sorteia uma palavra de uma lista ligada à computação.
2. A palavra começa oculta, com um traço para cada letra.
3. O jogador informa uma letra por vez e pode cometer até seis erros.
4. Um acerto revela todas as ocorrências daquela letra.
5. Uma letra válida que não pertence à palavra consome uma chance.
6. Letras repetidas e entradas inválidas não consomem chances.
7. Letras maiúsculas são convertidas para minúsculas; espaços nas extremidades são removidos.
8. O jogador vence ao descobrir todas as letras e perde ao atingir o limite de erros.
9. Ao terminar, o programa informa o resultado e revela a palavra.

Para manter o escopo adequado à atividade, o jogo aceita letras de **A a Z, sem acentos**. O banco de palavras segue essa regra. Não há cadastro, acesso à internet nem instalação de bibliotecas externas.


In [1]:
# Bibliotecas para demonstrar partidas e executar os testes.
import io
import sys
import unittest
from contextlib import redirect_stdout
from unittest.mock import patch

print("Python utilizado:", sys.version.split()[0])


Python utilizado: 3.13.3


## 3. Versão inicial

A célula abaixo contém a função completa da versão base, com os comentários do arquivo original.

A base usa a palavra fixa `python`, uma lista para guardar as tentativas e um `while` para manter a partida em execução. Dentro do laço, um `for` percorre a palavra e monta sua exibição. A mesma função concentra a entrada, as regras e as mensagens.

Essa versão permite jogar com entradas simples, mas apresenta limitações:

- A mesma letra errada desconta uma chance a cada repetição.
- Letras maiúsculas não são normalizadas.
- Números, símbolos e entradas com várias letras não são validados.
- A palavra é sempre a mesma.
- A mistura entre regras e interface dificulta testar partes isoladas.

Um detalhe interessante é que a entrada vazia passa pela condição `tentativa not in palavra` sem contar como erro, porque a string vazia está contida em qualquer string em Python. Mesmo assim, o programa não explica ao jogador que a entrada é inválida. Isso mostra como uma condição aparentemente simples pode não representar a regra desejada.

### Evidência do problema

Na execução da base com as entradas `b`, `b`, `p`, `y`, `t`, `h`, `o`, `n`, o contador passou de **1 para 2** após repetir `b`. O jogador acabou vencendo, mas recebeu duas penalidades pela mesma tentativa. Essa execução foi reproduzida no terminal com entradas controladas.

A lógica inicial está preservada na função `jogar_base`, para permitir a comparação com a solução final.


In [2]:
# VERSÃO BASE: apenas o nome da função foi adaptado para este notebook.
def jogar_base():
    """Executa uma partida com a palavra fixa "python" e até seis erros.

    Não recebe parâmetros. Lê as tentativas pelo teclado e imprime o resultado.
    Retorna None tanto na vitória quanto na derrota. Na versão final, jogar
    recebe a palavra e o limite de erros e retorna um bool com o resultado.

    Exemplo de uso em outro arquivo desta pasta:
        from forca_base import jogar
        jogar()  # Inicia a partida e aguarda a digitação do usuário.

    Nesta base, a função reúne exibição, entrada e regras do jogo. Na final,
    revelar_palavra e validar_letra assumem partes dessas responsabilidades.
    """
    # 1. CONFIGURAÇÃO E ESTADO INICIAL
    # A palavra é sempre a mesma. Na final, main() faz o sorteio e passa a
    # palavra para jogar(), permitindo também escolher uma palavra nos testes.
    palavra = "python"
    # A lista aceita duplicatas: duas entradas "b" ficam armazenadas duas vezes.
    # Na final, tentadas é um set e a validação avisa sobre letras repetidas.
    letras = []
    erros = 0

    # 2. REPETIÇÃO DA PARTIDA
    # O limite 6 está escrito diretamente no laço. Na final, é max_erros.
    # A vitória só será verificada dentro deste while, depois da exibição.
    while erros < 6:
        # 3. EXIBIÇÃO E VERIFICAÇÃO DE VITÓRIA
        # A exibição é reconstruída a cada rodada. Começamos supondo vitória;
        # encontrar qualquer letra ainda oculta muda ganhou para False.
        exibicao = ""
        ganhou = True
        for letra in palavra:
            if letra in letras:
                # Acerto: mostra a letra e um espaço para separar as posições.
                exibicao += letra + " "
            else:
                exibicao += "_ "
                ganhou = False

        # strip() remove o espaço que sobrou no final da exibição.
        # Na final, revelar_palavra() usa join() para montar essa mesma saída,
        # e issubset() verifica a vitória sem precisar da variável ganhou.
        print("Palavra:", exibicao.strip())
        if ganhou:
            print("Você venceu!")
            # Encerra a função imediatamente; o retorno implícito é None.
            return

        # 4. ENTRADA E ATUALIZAÇÃO DAS TENTATIVAS
        # Limitação: input() devolve o texto como foi digitado. "P" e "p" são
        # diferentes, e números ou várias letras não recebem aviso específico.
        # Na final, validar_letra() trata isso antes de registrar a tentativa.
        tentativa = input("Digite uma letra: ")
        letras.append(tentativa)
        # Limitação: repetir "b" desconta outra chance, pois não verificamos
        # se a letra já foi usada. Ex.: "b", "b" resulta em dois erros.
        # Além disso, "" e "py" são substrings de "python": não contam como
        # erro, embora não sejam tentativas válidas de uma única letra.
        if tentativa not in palavra:
            erros += 1
        print("Erros:", erros)

    # 5. ENCERRAMENTO
    # Só chegamos aqui quando o limite de erros é atingido. Na vitória, o
    # return acima já encerrou a função. Esta base não trata Ctrl+C ou EOF.
    print("Você perdeu! A palavra era:", palavra)


### Demonstração da base: erro repetido

O auxiliar abaixo fornece as letras na ordem indicada, como se alguém as digitasse. Ele também registra a saída para podermos fazer verificações. `patch` substitui `input` apenas durante a demonstração e restaura seu funcionamento ao terminar.

Digite a sequência `b`, `b`, `p`, `y`, `t`, `h`, `o`, `n` de forma simulada. Observe que a segunda tentativa de `b` aumenta o contador para **2**.


In [3]:
def simular_partida(funcao, entradas, *args, **kwargs):
    """Executa uma partida com entradas conhecidas e devolve resultado e saída."""
    entradas = iter(entradas)
    saida = io.StringIO()

    def entrada_simulada(mensagem):
        letra = next(entradas)
        print(mensagem + letra)  # Exibe também a entrada para facilitar a leitura.
        return letra

    with patch("builtins.input", side_effect=entrada_simulada):
        with redirect_stdout(saida):
            resultado = funcao(*args, **kwargs)
    return resultado, saida.getvalue()

sequencia_comparacao = ["b", "b", "p", "y", "t", "h", "o", "n"]
resultado_base, saida_base = simular_partida(jogar_base, sequencia_comparacao)
print(saida_base)
assert "Erros: 2" in saida_base
assert "Você venceu!" in saida_base
assert resultado_base is None


Palavra: _ _ _ _ _ _
Digite uma letra: b
Erros: 1
Palavra: _ _ _ _ _ _
Digite uma letra: b
Erros: 2
Palavra: _ _ _ _ _ _
Digite uma letra: p
Erros: 2
Palavra: p _ _ _ _ _
Digite uma letra: y
Erros: 2
Palavra: p y _ _ _ _
Digite uma letra: t
Erros: 2
Palavra: p y t _ _ _
Digite uma letra: h
Erros: 2
Palavra: p y t h _ _
Digite uma letra: o
Erros: 2
Palavra: p y t h o _
Digite uma letra: n
Erros: 2
Palavra: p y t h o n
Você venceu!



## 4. Roteiro de prompts estruturados

Os prompts abaixo são um roteiro reproduzível para orientar a construção, a revisão e a validação desta solução. Eles não são uma transcrição de conversas anteriores nem comprovantes de uso de outra ferramenta.

### Prompt 1 - Contexto e construção da base

```text
Atue como um parceiro de programação em Python para um estudante de Ciência
da Computação. Preciso desenvolver um jogo da forca de terminal para uma
atividade sobre colaboração entre humanos e IA.

Comece com uma versão básica usando a palavra fixa "python", uma lista de
tentativas e até seis erros. Mostre as letras descobertas, informe vitória
ou derrota e use apenas a biblioteca padrão. Explique o funcionamento e
aponte as limitações dessa primeira versão, sem tratá-la como produto pronto.
```

Esse pedido delimita o problema e o nível de complexidade. Também pede uma análise das limitações, para que a primeira resposta seja um ponto de partida.

### Prompt 2 - Refatoração com critérios verificáveis

```text
Revise a versão inicial em forca_base.py e produza uma versão refatorada.
Separe a exibição da palavra, a validação da letra e o controle da partida
em funções. Use um conjunto para armazenar tentativas sem duplicação.

Regras: aceite uma letra de A a Z sem acento, normalize maiúsculas e espaços
nas extremidades, rejeite entradas vazias e não desconte chances por letras
repetidas ou entradas inválidas. Revele todas as ocorrências de um acerto.
Sorteie a palavra de uma lista e mostre as tentativas e os erros na tela.

Mantenha o código compreensível para um estudante. Explique quais alterações
são de organização e quais corrigem ou acrescentam comportamentos.
```

Aqui o pedido deixa de ser apenas “melhore esse código”. As regras permitem conferir se a alteração resolveu o problema. Separar funções é uma refatoração; mudar o tratamento de entradas e sortear palavras são correções e melhorias de comportamento.

### Prompt 3 - Depuração de um problema concreto

```text
Na versão inicial, a palavra é "python". Ao digitar "b" duas vezes, o
contador de erros passa de 0 para 1 e depois para 2. Pela regra definida,
uma letra repetida deve gerar um aviso e manter o contador em 1.

Identifique a causa e organize a validação antes da atualização do estado.
Verifique também se " B " é reconhecida como a mesma tentativa.
```

Esse prompt informa entrada, resultado observado e resultado esperado. A causa está na ausência de uma verificação de repetição antes de incrementar os erros. Na solução final, a entrada é normalizada e validada antes de entrar no conjunto de tentativas.

### Prompt 4 - Validação da solução

```text
Crie testes com unittest para as regras do jogo e para partidas completas.
Cubra vitória, derrota após seis erros, letra repetida, entrada vazia,
números, acentos, maiúsculas e palavras com letras repetidas.

Use entradas controladas para que os testes não dependam de digitação nem
de sorteio. Verifique também o encerramento por fim de entrada ou Ctrl+C.
Execute os testes e informe o resultado real, sem inventar evidências.
```


## 5. Solução final e decisões de implementação

O código completo da versão final está dividido nas próximas células por responsabilidade. Execute todas elas antes de iniciar uma partida.

| Parte | Responsabilidade | Motivo da escolha |
|---|---|---|
| `revelar_palavra` | Montar a palavra com letras descobertas e traços | Permite testar a exibição sem iniciar uma partida. |
| `validar_letra` | Normalizar e verificar a entrada | Evita alterar o jogo com uma tentativa inválida. |
| `jogar` | Controlar tentativas, erros e resultado | Reúne as regras e recebe uma palavra conhecida para os testes. |
| `main` | Sortear a palavra e tratar o encerramento | Separa o início do programa da execução de uma partida. |

As tentativas ficam em um `set`, que é uma coleção sem elementos duplicados. A função de validação continua necessária: o conjunto evita armazenar duplicatas, mas sozinho não impede que o código desconte outra chance.

Para descobrir a vitória, o programa verifica se todas as letras distintas da palavra estão entre as tentadas, usando `issubset`. Na palavra `dados`, por exemplo, basta acertar `d`, `a`, `o` e `s`: a letra `d` aparece duas vezes na tela, mas precisa ser informada apenas uma vez.

A exibição usa `join` para juntar letras e traços com espaços. Como conjuntos não garantem uma ordem de apresentação apropriada para o jogador, o histórico é mostrado em ordem alfabética com `sorted`.

### Comparação entre as versões

| Critério | Versão inicial | Versão final |
|---|---|---|
| Palavra | Fixa: `python` | Sorteada de seis opções |
| Estrutura | Uma função concentra tudo | Funções com responsabilidades definidas |
| Letras tentadas | Lista com possíveis repetições | Conjunto e validação de repetição |
| Entrada inválida | Sem aviso específico | Mensagem e nova tentativa sem penalidade |
| Maiúsculas e espaços externos | Não tratados | Normalizados antes da comparação |
| Erro repetido | Consome outra chance | Mostra aviso e mantém o contador |
| Exibição | Palavra e erros | Palavra, limite de erros e histórico |
| Validação automatizada | Base preservada como comparação | 16 testes automatizados |

Para este tamanho de jogo, considero a clareza e a correção mais importantes do que buscar ganhos de velocidade. O conjunto também facilita consultas, mas não foi feito benchmark; portanto, não atribuo um percentual de desempenho à refatoração.


### 5.1. Configuração e banco de palavras

Na base, `python` e o limite de seis erros estavam diretamente na função. Aqui, as constantes reúnem os padrões do jogo. `random.choice` será usado para o sorteio e `ascii_lowercase` define o alfabeto aceito.


In [4]:
import random
from string import ascii_lowercase

# Palavras sem acentos, compatíveis com as regras de validação.
PALAVRAS = ("python", "algoritmo", "dados", "variavel", "teclado", "computador")
MAX_ERROS = 6


### 5.2. Exibir a palavra

Esta função substitui a concatenação manual da base. Ela retorna um texto e não altera as tentativas.


In [5]:
def revelar_palavra(palavra: str, tentadas: set[str]) -> str:
    """Recebe a palavra e as letras tentadas; devolve sua exibição como texto.

    Não imprime nem modifica o conjunto recebido, facilitando testes isolados.
    Substitui a montagem manual de exibicao dentro do while da versão base.

    Exemplos:
        >>> revelar_palavra("dados", set())
        '_ _ _ _ _'
        >>> revelar_palavra("dados", {"d"})
        'd _ d _ _'
    """
    # 3. EXIBIÇÃO: para cada posição, escolhe a letra descoberta ou um traço.
    # join() une esses elementos com espaços, revelando também letras repetidas.
    return " ".join(letra if letra in tentadas else "_" for letra in palavra)


### 5.3. Validar a tentativa

Esta função normaliza a entrada e impede que letras repetidas ou inválidas alterem o contador de erros.


In [6]:
def validar_letra(entrada: str, tentadas: set[str]) -> str:
    """Recebe o texto digitado e o histórico; retorna uma letra válida e inédita.

    Lança ValueError se a entrada não for uma única letra de A a Z sem acento
    ou se ela já tiver sido tentada. Não acrescenta a letra ao conjunto.
    Na base, esse cuidado não existia: a entrada ia diretamente para a lista.

    Exemplos:
        >>> validar_letra(" A ", set())
        'a'
        >>> validar_letra("B", {"a"})
        'b'

    validar_letra("a", {"a"}) gera ValueError por repetição.
    validar_letra("12", set()) gera ValueError por formato inválido.
    """
    # 4. VALIDAÇÃO: strip() remove espaços externos; lower() converte para
    # minúsculas. Assim, " A " e "a" são reconhecidas como a mesma tentativa.
    letra = entrada.strip().lower()
    # É preciso conferir o tamanho e o alfabeto permitido. Apenas isalpha()
    # também aceitaria letras acentuadas, que estão fora das regras deste jogo.
    if len(letra) != 1 or letra not in ascii_lowercase:
        raise ValueError("Digite apenas uma letra de A a Z, sem acento.")
    # Normalizar antes desta comparação impede repetir "a" digitando "A".
    if letra in tentadas:
        raise ValueError("Você já tentou essa letra. Escolha outra.")
    return letra


### 5.4. Controlar a partida

Esta é a função principal das regras. Compare seus blocos numerados com os da base: configuração, repetição, exibição, entrada e encerramento.


In [7]:
def jogar(palavra: str, max_erros: int = MAX_ERROS) -> bool:
    """Controla uma partida pelo terminal e retorna True na vitória ou False na derrota.

    palavra: texto não vazio, apenas com letras minúsculas de a a z.
    max_erros: inteiro positivo; se omitido, usa o padrão de seis erros.
    Configurações inválidas geram ValueError antes de começar a partida.

    Exemplos de uso interativo (aguardam digitação):
        jogar("dados")               # Digite d, a, o, s para vencer.
        jogar("python", max_erros=3) # A derrota ocorre ao cometer três erros.

    Na base, jogar() não recebia argumentos e não devolvia o resultado.
    Receber a palavra permite testar partidas sem depender do sorteio.
    """
    # 1. CONFIGURAÇÃO E ESTADO INICIAL
    # Como agora a palavra pode vir de quem chama a função, validamos se ela
    # respeita o mesmo alfabeto aceito pela entrada do jogador.
    if not palavra or any(letra not in ascii_lowercase for letra in palavra):
        raise ValueError("A palavra deve conter apenas letras minúsculas de a a z.")
    # A comparação exata de tipo também rejeita True/False: em Python, bool
    # é uma subclasse de int, mas não representa um limite adequado aqui.
    if type(max_erros) is not int or max_erros < 1:
        raise ValueError("O limite de erros deve ser um inteiro positivo.")

    # O set guarda valores únicos. Ex.: set("dados") contém d, a, o e s.
    # tentadas guarda acertos e erros; letras_da_palavra guarda o alvo da vitória.
    tentadas: set[str] = set()
    letras_da_palavra = set(palavra)
    erros = 0

    print("\n=== JOGO DA FORCA ===")
    print(f"Descubra a palavra. Você pode errar até {max_erros} vezes.")
    print("Digite uma letra por vez, sem acento. Use Ctrl+C para sair.")

    # 2. REPETIÇÃO DA PARTIDA
    # Continua enquanto há chances E ainda faltam letras. issubset() verifica
    # se todas as letras da palavra estão nas tentadas, substituindo a variável
    # ganhou que a versão base recalculava durante a montagem da exibição.
    while erros < max_erros and not letras_da_palavra.issubset(tentadas):
        # 3. EXIBIÇÃO: a função extraída substitui o for de concatenação da base.
        print("\nPalavra:", revelar_palavra(palavra, tentadas))
        print(f"Erros: {erros}/{max_erros}")
        # sorted() organiza o conjunto alfabeticamente. Se estiver vazio,
        # join() produz "" e o operador or seleciona a mensagem "nenhuma".
        print("Letras tentadas:", ", ".join(sorted(tentadas)) or "nenhuma")

        # 4. ENTRADA E ATUALIZAÇÃO DAS TENTATIVAS
        # A tentativa só entra no estado do jogo depois de passar na validação.
        try:
            letra = validar_letra(input("Sua letra: "), tentadas)
        except ValueError as erro:
            print(erro)
            # Volta ao início do while sem registrar a letra nem aumentar erros.
            # Corrige a base, que penalizava novamente uma letra errada repetida.
            continue

        # Só uma letra válida e inédita pode alterar o estado da partida.
        # O set sozinho não resolveria a penalidade duplicada: a validação acima
        # é o que impede uma tentativa repetida de chegar ao contador de erros.
        tentadas.add(letra)
        if letra in letras_da_palavra:
            print("Boa! Essa letra está na palavra.")
        else:
            erros += 1
            print("Essa letra não está na palavra.")

    # 5. ENCERRAMENTO
    # O laço terminou por vitória ou limite de erros. A mesma condição de
    # conjunto identifica qual dos dois aconteceu, sem depender da última letra.
    venceu = letras_da_palavra.issubset(tentadas)
    print("\nPalavra:", revelar_palavra(palavra, tentadas))
    print(f"Erros: {erros}/{max_erros}")
    if venceu:
        print(f"Você venceu! A palavra era: {palavra}")
    else:
        print(f"Você perdeu! A palavra era: {palavra}")
    # Além da mensagem ao jogador, fornece um resultado verificável nos testes.
    return venceu


### 5.5. Iniciar e encerrar o programa

O sorteio fica separado da partida. Para exemplos e testes, podemos escolher uma palavra conhecida chamando jogar diretamente.


In [8]:
def main() -> None:
    """Inicia uma partida com palavra sorteada e trata a saída do usuário.

    Não recebe argumentos e retorna None. É chamada ao executar o arquivo.
    Também pode ser usada com `from forca import main; main()` em um console
    Python aberto nesta pasta; a partida aguardará entradas pelo teclado.

    Na base, jogar() era chamada diretamente e não havia sorteio nem tratamento
    de interrupção. Aqui, jogar() continua disponível para partidas controladas.
    """
    try:
        # O sorteio fica fora de jogar(): nos testes podemos usar sempre "dados".
        jogar(random.choice(PALAVRAS))
    except (EOFError, KeyboardInterrupt):
        # EOFError: terminou a entrada disponível. KeyboardInterrupt: Ctrl+C.
        # Trata esses encerramentos com uma mensagem, sem exibir traceback.
        print("\nPartida encerrada. Até a próxima!")


### 5.6. Exemplos rápidos das funções

As anotações de tipo ajudam a ler o código: `str` indica texto, `set[str]` indica um conjunto de textos e `bool` indica `True` ou `False`. Elas não fazem a validação automaticamente.

Neste exemplo, `d` revela duas posições de `dados`; ` A ` é transformada em `a`; e a repetição gera um aviso.


In [9]:
print("Palavra oculta:", revelar_palavra("dados", set()))
print("Acerto de d:", revelar_palavra("dados", {"d"}))
print("Entrada normalizada:", repr(validar_letra(" A ", set())))

try:
    validar_letra("A", {"a"})
except ValueError as erro:
    print("Tentativa repetida:", erro)

assert revelar_palavra("dados", {"d"}) == "d _ d _ _"
assert validar_letra(" A ", set()) == "a"


Palavra oculta: _ _ _ _ _
Acerto de d: d _ d _ _
Entrada normalizada: 'a'
Tentativa repetida: Você já tentou essa letra. Escolha outra.


### 5.7. Comparação com as mesmas entradas

Agora a versão final recebe exatamente a palavra e as tentativas usadas na base. A segunda entrada `b` é rejeitada por repetição e o jogo termina com **1 erro**, mostrando a correção do comportamento.


In [10]:
resultado_final, saida_final = simular_partida(
    jogar, sequencia_comparacao, "python"
)
print(saida_final)
assert resultado_final is True
assert "Você já tentou essa letra" in saida_final
assert "Erros: 1/6" in saida_final
assert "Erros: 2/6" not in saida_final
print("Comparação conferida: base = 2 erros; versão final = 1 erro.")



=== JOGO DA FORCA ===
Descubra a palavra. Você pode errar até 6 vezes.
Digite uma letra por vez, sem acento. Use Ctrl+C para sair.

Palavra: _ _ _ _ _ _
Erros: 0/6
Letras tentadas: nenhuma
Sua letra: b
Essa letra não está na palavra.

Palavra: _ _ _ _ _ _
Erros: 1/6
Letras tentadas: b
Sua letra: b
Você já tentou essa letra. Escolha outra.

Palavra: _ _ _ _ _ _
Erros: 1/6
Letras tentadas: b
Sua letra: p
Boa! Essa letra está na palavra.

Palavra: p _ _ _ _ _
Erros: 1/6
Letras tentadas: b, p
Sua letra: y
Boa! Essa letra está na palavra.

Palavra: p y _ _ _ _
Erros: 1/6
Letras tentadas: b, p, y
Sua letra: t
Boa! Essa letra está na palavra.

Palavra: p y t _ _ _
Erros: 1/6
Letras tentadas: b, p, t, y
Sua letra: h
Boa! Essa letra está na palavra.

Palavra: p y t h _ _
Erros: 1/6
Letras tentadas: b, h, p, t, y
Sua letra: o
Boa! Essa letra está na palavra.

Palavra: p y t h o _
Erros: 1/6
Letras tentadas: b, h, o, p, t, y
Sua letra: n
Boa! Essa letra está na palavra.

Palavra: p y t h o n
Err

## 6. Validação: entradas inválidas, repetição e vitória

A sequência `1`, `x`, `x`, `D`, `a`, `o`, `s` permite verificar três regras na mesma partida: rejeição de números, ausência de penalidade por repetição e normalização de maiúsculas. A palavra é `dados` e o resultado esperado é vitória com apenas um erro.


In [11]:
resultado_demo, saida_demo = simular_partida(
    jogar, ["1", "x", "x", "D", "a", "o", "s"], "dados"
)
print(saida_demo)
assert resultado_demo is True
assert "Palavra: d a d o s" in saida_demo
assert "Erros: 1/6" in saida_demo
assert "Erros: 2/6" not in saida_demo



=== JOGO DA FORCA ===
Descubra a palavra. Você pode errar até 6 vezes.
Digite uma letra por vez, sem acento. Use Ctrl+C para sair.

Palavra: _ _ _ _ _
Erros: 0/6
Letras tentadas: nenhuma
Sua letra: 1
Digite apenas uma letra de A a Z, sem acento.

Palavra: _ _ _ _ _
Erros: 0/6
Letras tentadas: nenhuma
Sua letra: x
Essa letra não está na palavra.

Palavra: _ _ _ _ _
Erros: 1/6
Letras tentadas: x
Sua letra: x
Você já tentou essa letra. Escolha outra.

Palavra: _ _ _ _ _
Erros: 1/6
Letras tentadas: x
Sua letra: D
Boa! Essa letra está na palavra.

Palavra: d _ d _ _
Erros: 1/6
Letras tentadas: d, x
Sua letra: a
Boa! Essa letra está na palavra.

Palavra: d a d _ _
Erros: 1/6
Letras tentadas: a, d, x
Sua letra: o
Boa! Essa letra está na palavra.

Palavra: d a d o _
Erros: 1/6
Letras tentadas: a, d, o, x
Sua letra: s
Boa! Essa letra está na palavra.

Palavra: d a d o s
Erros: 1/6
Você venceu! A palavra era: dados



### 6.1. Testes automatizados completos

As classes abaixo reproduzem os **16 testes** de `test_forca.py`, usando diretamente as funções deste notebook. Há testes de exibição, validação, vitória, derrota, repetição, limites e encerramento. Alguns verificam várias entradas com `subTest`.

`assertEqual` compara valores; `assertTrue`/`assertFalse` verificam resultados; `assertRaises` verifica um erro esperado. As entradas são controladas, sem sorteio nem digitação.

A célula seguinte executa somente essas classes com `TextTestRunner`. Isso evita que `unittest.main()` interprete os argumentos internos do Jupyter.


In [12]:
"""Testes das regras e de partidas completas, com entradas reproduzíveis."""

import io
import unittest
from contextlib import redirect_stdout
from unittest.mock import patch

# Usa jogar, main, revelar_palavra e validar_letra definidos nas células anteriores.


class TestRevelarPalavra(unittest.TestCase):
    def test_comeca_oculta(self):
        self.assertEqual(revelar_palavra("dados", set()), "_ _ _ _ _")

    def test_revela_todas_as_ocorrencias(self):
        self.assertEqual(revelar_palavra("dados", {"d"}), "d _ d _ _")

    def test_palavra_completa(self):
        self.assertEqual(revelar_palavra("dados", set("dados")), "d a d o s")


class TestValidarLetra(unittest.TestCase):
    def test_normaliza_maiuscula_e_espacos(self):
        self.assertEqual(validar_letra(" A ", set()), "a")

    def test_rejeita_entradas_invalidas(self):
        for entrada in ("", "  ", "ab", "1", "!", "á", "ç", "🙂"):
            with self.subTest(entrada=entrada), self.assertRaises(ValueError):
                validar_letra(entrada, set())

    def test_rejeita_repetida_depois_de_normalizar(self):
        with self.assertRaisesRegex(ValueError, "já tentou"):
            validar_letra(" A ", {"a"})

    def test_validacao_nao_modifica_tentadas(self):
        tentadas = {"a"}
        validar_letra("b", tentadas)
        self.assertEqual(tentadas, {"a"})


class TestPartida(unittest.TestCase):
    def executar_partida(self, palavra, entradas, max_erros=6):
        saida = io.StringIO()
        with patch("builtins.input", side_effect=entradas) as ler:
            with redirect_stdout(saida):
                venceu = jogar(palavra, max_erros)
        return venceu, saida.getvalue(), ler.call_count

    def test_vitoria_com_letras_repetidas_na_palavra(self):
        venceu, saida, leituras = self.executar_partida("dados", ["d", "a", "o", "s"])
        self.assertTrue(venceu)
        self.assertEqual(leituras, 4)
        self.assertIn("Você venceu! A palavra era: dados", saida)
        self.assertIn("Palavra: d a d o s", saida)

    def test_derrota_apos_exatamente_seis_erros(self):
        venceu, saida, leituras = self.executar_partida("a", list("bcdefg"))
        self.assertFalse(venceu)
        self.assertEqual(leituras, 6)
        self.assertIn("Erros: 6/6", saida)
        self.assertIn("Você perdeu! A palavra era: a", saida)

    def test_entradas_invalidas_nao_consumem_chances(self):
        venceu, saida, _ = self.executar_partida("a", ["", "ab", "1", "!", "á", " A "], 1)
        self.assertTrue(venceu)
        self.assertIn("Erros: 0/1", saida)

    def test_repetir_erro_nao_desconta_outra_chance(self):
        venceu, saida, leituras = self.executar_partida("a", ["b", " B ", "a"], 2)
        self.assertTrue(venceu)
        self.assertEqual(leituras, 3)
        self.assertIn("Você já tentou essa letra", saida)
        self.assertIn("Erros: 1/2", saida)

    def test_repetir_acerto_nao_desconta_chance(self):
        venceu, saida, _ = self.executar_partida("ab", ["a", "a", "b"], 1)
        self.assertTrue(venceu)
        self.assertIn("Erros: 0/1", saida)

    def test_vence_apos_cinco_erros(self):
        venceu, saida, _ = self.executar_partida("a", list("bcdefa"))
        self.assertTrue(venceu)
        self.assertIn("Erros: 5/6", saida)

    def test_rejeita_palavra_incompativel_com_as_regras(self):
        for palavra in ("", "Python", "ação", "duas palavras", "123"):
            with self.subTest(palavra=palavra), self.assertRaises(ValueError):
                jogar(palavra)

    def test_rejeita_limite_invalido(self):
        for limite in (0, -1, 1.5, True, "6"):
            with self.subTest(limite=limite), self.assertRaises(ValueError):
                jogar("python", limite)


class TestEncerramento(unittest.TestCase):
    def test_fim_da_entrada_e_interrupcao(self):
        for erro in (EOFError, KeyboardInterrupt):
            with self.subTest(erro=erro.__name__):
                saida = io.StringIO()
                with patch("builtins.input", side_effect=erro), redirect_stdout(saida):
                    main()
                self.assertIn("Partida encerrada", saida.getvalue())


In [13]:
# Cria uma suíte nova em cada execução, permitindo repetir esta célula.
classes_de_teste = (
    TestRevelarPalavra, TestValidarLetra, TestPartida, TestEncerramento
)
suite = unittest.TestSuite(
    unittest.defaultTestLoader.loadTestsFromTestCase(classe)
    for classe in classes_de_teste
)
resultado_testes = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert resultado_testes.testsRun == 16
assert resultado_testes.wasSuccessful(), "Revise os testes que falharam acima."
print("Validação concluída: 16 testes aprovados.")


test_comeca_oculta (__main__.TestRevelarPalavra.test_comeca_oculta) ... 

ok


test_palavra_completa (__main__.TestRevelarPalavra.test_palavra_completa) ... 

ok


test_revela_todas_as_ocorrencias (__main__.TestRevelarPalavra.test_revela_todas_as_ocorrencias) ... 

ok


... (24 outputs left out)

test_fim_da_entrada_e_interrupcao (__main__.TestEncerramento.test_fim_da_entrada_e_interrupcao) ... 

ok


----------------------------------------------------------------------
Ran 16 tests in 0.037s

OK


Validação concluída: 16 testes aprovados.


## 7. Partida interativa opcional

Para jogar manualmente, execute primeiro as células de definição das funções. Depois, na célula abaixo, altere `JOGAR_INTERATIVAMENTE` para `True`, escolha `"base"` ou `"final"` em `VERSAO` e execute **somente essa célula**. Digite uma letra a cada solicitação.

O valor inicial é `False` para que **Executar tudo** termine sem aguardar entradas. Para sair de uma partida no notebook, use a ação de interromper a execução do seu ambiente. Antes de executar tudo novamente, volte a opção para `False`.


In [14]:
JOGAR_INTERATIVAMENTE = False
VERSAO = "final"  # Opções: "base" ou "final".

if JOGAR_INTERATIVAMENTE:
    if VERSAO == "base":
        jogar_base()
    elif VERSAO == "final":
        main()
    else:
        raise ValueError('Escolha VERSAO = "base" ou VERSAO = "final".')
else:
    print("Para jogar, altere JOGAR_INTERATIVAMENTE para True e execute esta célula.")


Para jogar, altere JOGAR_INTERATIVAMENTE para True e execute esta célula.


## 8. Análise crítica dos exemplos de refatoração do material

### Soma de lista

Trocar um `while` com índice por `sum(numeros)` deixa a intenção mais clara e elimina o controle manual do índice. Para uma lista de inteiros, ambas as versões percorrem os elementos e têm complexidade de tempo **O(n)**. Portanto, reduzir o número de linhas não significa mudar automaticamente a ordem de complexidade.


In [15]:
numeros = [10, 20, -5]
soma = sum(numeros)
assert soma == 25
assert sum([]) == 0

print("Exemplo executado: verificações aprovadas.")


Exemplo executado: verificações aprovadas.


### Verificação de número primo

O material mostra a redução da busca por divisores até a raiz quadrada do número. A ideia funciona porque, se um número composto é escrito como produto de dois fatores, pelo menos um deles é menor ou igual à sua raiz quadrada.

Porém, as versões apresentadas precisam de uma verificação para `n < 2`. Sem ela, `0` e `1` acabam classificados como primos; na versão com `sqrt`, números negativos ainda podem provocar erro. A solução abaixo usa `isqrt`, que calcula a raiz quadrada inteira sem conversão para ponto flutuante.


In [16]:
from math import isqrt


def eh_primo(n: int) -> bool:
    if n < 2:
        return False
    for divisor in range(2, isqrt(n) + 1):
        if n % divisor == 0:
            return False
    return True


assert not eh_primo(-7)
assert not eh_primo(0)
assert not eh_primo(1)
assert eh_primo(2)
assert eh_primo(97)
assert not eh_primo(49)

print("Exemplo executado: verificações aprovadas.")


Exemplo executado: verificações aprovadas.


A função recebe números inteiros. A contagem de candidatos a divisor cai de **O(n)** para **O(√n)** no pior caso, considerando cada operação aritmética com custo constante. O `+ 1` é necessário para incluir a raiz inteira no teste e identificar quadrados perfeitos, como `49`.

### Inversão de string

O fatiamento `texto[::-1]` expressa a inversão de forma curta e legível. A versão que coloca cada letra no início de uma nova string copia trechos progressivamente maiores, tendo custo quadrático; o fatiamento percorre o texto com custo linear e produz uma nova string.


In [17]:
def inverte_texto(texto: str) -> str:
    return texto[::-1]


assert inverte_texto("python") == "nohtyp"
assert inverte_texto("") == ""
assert inverte_texto("a") == "a"

print("Exemplo executado: verificações aprovadas.")


Exemplo executado: verificações aprovadas.


Essa inversão opera sobre os pontos de código da string. Ela atende aos exemplos simples da aula, mas não resolve sozinha a inversão visual de todos os emojis compostos ou caracteres com marcas combinantes.

Esses exemplos mostram por que a revisão continua necessária: um código pode ficar menor ou mais eficiente e ainda ter um erro nas condições de entrada.


## 9. Reflexão sobre a colaboração humano-IA

### Em que a IA contribuiu?

A IA apoiou a criação das versões, a separação das funções, o tratamento de entradas e a elaboração dos testes. Ela também ajudou a transformar regras escritas em comportamentos verificáveis, como não penalizar uma letra repetida.

### O que precisa continuar sob decisão humana?

É necessário definir as regras, avaliar se as sugestões fazem sentido e compreender o que será entregue. Por exemplo, aceitar acentos ou permitir que o jogador tente a palavra inteira são decisões do projeto, não algo que a IA deveria presumir sem contexto.

Minha avaliação é que a regra de repetição merece atenção especial. Mesmo utilizando um conjunto, o erro continuaria existindo se o contador fosse atualizado antes da validação. A escolha da estrutura de dados não substitui a análise da ordem das operações.

### Qual foi a melhoria mais relevante?

Considero a separação entre validar e alterar o estado a melhoria mais relevante. Primeiro o programa verifica a entrada; somente depois registra a tentativa e decide se aumenta os erros. Isso torna a regra mais fácil de entender e evita penalidades indevidas.

### Quais são os limites da solução?

O banco contém apenas seis palavras, não há níveis de dificuldade, pontuação persistente ou interface gráfica. A validação aceita apenas letras sem acento e não permite tentar a palavra inteira. Essas limitações são compatíveis com a proposta de um miniprojeto de terminal.

Uma evolução possível seria separar também toda a entrada e saída das regras, permitindo reaproveitar o jogo em outra interface. Para esta entrega, funções pequenas e testes de partidas foram suficientes para manter o código compreensível.

### O que levo da atividade?

Para mim, a principal conclusão é que usar IA para programar exige saber avaliar o resultado. Um prompt estruturado ajuda, mas a confiança no código precisa vir da compreensão das regras e da validação. O exemplo de números primos reforça isso: otimizar o laço sem tratar `0` e `1` deixaria a solução incorreta.


## 10. Conclusão da entrega

O notebook reúne a base e a versão final, os comentários de funcionamento, um roteiro de prompts, comparações com entradas iguais e testes executáveis. A refatoração deixou as responsabilidades mais claras; as correções fizeram o jogo respeitar as regras de validação e repetição.

A versão inicial continua disponível como `jogar_base`, e a versão final como `jogar`. As saídas das demonstrações e dos testes são produzidas pelas próprias células, permitindo reproduzir a comparação.

Para entregar a atividade, este arquivo `.ipynb` contém todo o código necessário. O relatório Markdown e os arquivos `.py` continuam no repositório como materiais complementares.


## 11. Material de referência

VALÉRIA, Kadidja. **Programação Assistida por Inteligência Artificial: o novo paradigma da colaboração humano-máquina.** Material da disciplina, UDF, disponibilizado em [Collaborative_AI_Programming_AtividadePrática.pdf](Collaborative_AI_Programming_AtividadePrática.pdf).

Foram utilizados o ciclo de colaboração da página 4, os exemplos de refatoração das páginas 5 a 7, o desafio de miniprojeto da página 9, o roteiro de prompts da página 10 e a discussão sobre responsabilidade das páginas 11 e 12.
